# RQ2 — Preflight offline del piano D/C/K

Dopo aver superato i **15 test sintetici PyCharm** del modulo `test_rq2_dryrun_plan.py`, eseguire la sola cella Python. Verifica la coorte V1 contro i due CSV GT-free e gli SHA congelati, senza aprire Market.zip/record.csv/scoring o invocare provider. Visualizza **solo il sommario** del piano non eseguibile.

Attesi 4 discovery / 11 test; 90 stage D, 90 C, 24 K, totale 204 stage logici. **Non sono 204 chiamate API.** Tutte le autorizzazioni ai modelli restano disattivate, e non viene scritto alcun file di piano.


In [ ]:
# RQ2 — STAGE PLAN PREFLIGHT GT-FREE: no calls, no saved plan.
# 204 STAGE ENTRIES are NOT 204 API calls. Each D stage may consume many LLM calls.
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys

BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
ROOT = Path("/content/drive/MyDrive/Tesi_OpenRCA")
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
for rel in (
    "data/processed/split_market_v1.csv",
    "data/processed/benchmark_inputs_market.csv",
):
    if not (ROOT / rel).is_file():
        raise FileNotFoundError("Missing frozen GT-free input: " + rel)

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Unexpected non-Git worktree: no deletion or overwrite")
    actual_branch = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True,
    ).strip()
    if actual_branch != BRANCH:
        raise RuntimeError("Wrong Git branch; no automatic checkout")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True,
    ).strip():
        raise RuntimeError("Local changes: refusing pull")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        capture_output=True, check=True, text=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         BRANCH, URL, str(REPO)],
        capture_output=True, check=True, text=True,
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True,
).strip()
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
res = subprocess.run(
    [sys.executable, "-m", "evaluation.rq2_dryrun_plan",
     "--data-dir", str(ROOT)],
    cwd=REPO, capture_output=True, text=True, env=env,
)
if res.returncode:
    raise RuntimeError("GT-free workflow preflight failed:\n" + res.stderr[-5000:])
try:
    result = json.loads(res.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("No valid RQ2 JSON summary") from exc

expected = (
    result.get("protocol") == "rq2_dryrun_workflow_preview_v0"
    and result.get("status") == "DRAFT_NOT_EXECUTABLE"
    and result.get("source_verified") is True
    and result.get("stages") == {"D":90, "C":90, "K":24}
    and result.get("total_stage_entries") == 204
    and result.get("partition", {}).get("discovery_cases") == 4
    and result.get("partition", {}).get("test_cases") == 11
    and result.get("partition", {}).get("discovery_windows") == 4
    and result.get("partition", {}).get("test_windows") == 10
    and result.get("cross_phase_gate") == "TEST_BLOCKED_UNTIL_DISCOVERY_RULES_FREEZE"
    and result.get("can_execute_api") is False
    and result.get("api_budget_approved") is False
    and result.get("model_api_calls") == 0
    and result.get("files_written") is False
    and result.get("reads_ground_truth") is False
)
if not expected:
    raise RuntimeError("Draft preflight result violated protocol safeguards")
print("=== RQ2 — GT-FREE DRY-RUN STAGE PREFLIGHT ===")
print("Branch:", BRANCH)
print("Commit:", commit)
print(json.dumps(result, indent=2, ensure_ascii=False))
print("\nNO MODEL API CALLS. NO RUN PLAN SAVED. NO RQ1 MODIFICATIONS.")


## Interpretazione

`DRAFT_NOT_EXECUTABLE` vuol dire che il protocollo, la selezione e il budget non sono ancora congelati. Le fasi test resteranno bloccate fino a freeze delle regole candidate; questa protezione dovrà essere verificata anche nel futuro runner reale. Riferire output completo in chat per revisione.
